<a href="https://colab.research.google.com/github/edi-oliveira1979/mestrado-cesar-school/blob/main/Mestrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração de Ambiente e Segurança
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os, json, math
from google.colab import userdata

# 1. Configuração de Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Autenticação Kaggle (Segurança via Secrets)
def setup_kaggle():
    k_user = userdata.get('KAGGLE_USERNAME').strip()
    k_key = userdata.get('KAGGLE_KEY').strip()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": k_user, "key": k_key}, f)
    os.chmod('/root/.kaggle/kaggle.json', 600)

setup_kaggle()
print(f"✅ Ambiente configurado. Dispositivo: {device}")

In [ ]:
# Célula 2: Download e Carga de Dados
# 1. Download do arquivo principal
!kaggle competitions download -c home-credit-default-risk -f application_train.csv --force

# 2. Extração (Correção do erro BadGzipFile/PK)
if os.path.exists('application_train.csv'):
    !mv application_train.csv application_train.zip
    !unzip -o application_train.zip
    !rm application_train.zip

# 3. Carga inicial
df_raw = pd.read_csv('application_train.csv')
print(f"✅ Dados carregados: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")

In [ ]:
# Célula 3: Engenharia de Targets (Alvos)
# Criar scalers para os alvos de regressão (Prescrição)
scaler_credit = StandardScaler()
scaler_annuity = StandardScaler()

df_raw['TARGET_CREDIT_SCALED'] = scaler_credit.fit_transform(df_raw[['AMT_CREDIT']])
df_raw['TARGET_ANNUITY_SCALED'] = scaler_annuity.fit_transform(df_raw[['AMT_ANNUITY']])

# Identificamos os alvos oficiais para o modelo MTL
targets_df = df_raw[['TARGET', 'TARGET_CREDIT_SCALED', 'TARGET_ANNUITY_SCALED']]
print("✅ Alvos (Targets) normalizados e preparados.")

In [ ]:
# Célula 4: Purificação e Limpeza de Features
# 1. Lista de Colunas a Excluir (Vazamento e Identificadores)
drop_cols = [
    'SK_ID_CURR', 'TARGET',
    'AMT_CREDIT', 'AMT_ANNUITY',
    'TARGET_CREDIT_SCALED', 'TARGET_ANNUITY_SCALED',
    'AMT_GOODS_PRICE' # Removido por ser redundante ao crédito
]

# 2. Separação de Features X
X = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns])

# 3. Identificação de Tipos e Imputação
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

for col in num_cols: X[col] = X[col].fillna(X[col].median())
for col in cat_cols: X[col] = X[col].fillna('NA_Value')

print(f"✅ Features purificadas: {X.shape[1]} colunas (Num: {len(num_cols)}, Cat: {len(cat_cols)})")

In [ ]:
# Célula 5: Encoding e Escalonamento Final
# 1. Label Encoding para Categóricos
cat_cardinalities = []
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    cat_cardinalities.append(len(le.classes_))

# 2. Standard Scaling para Numéricos
scaler_X = StandardScaler()
X[num_cols] = scaler_X.fit_transform(X[num_cols])

# 3. Split Treino/Validação (80/20)
X_train_num, X_val_num, X_train_cat, X_val_cat, y_train, y_val = train_test_split(
    X[num_cols], X[cat_cols], targets_df, test_size=0.2, random_state=42, stratify=targets_df['TARGET']
)

print(f"✅ Divisão de dados concluída. Treino: {len(X_train_num)} amostras.")

In [ ]:
# Célula 6: Arquitetura FT-Transformer MTL
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num, cat_cards, d_emb):
        super().__init__()
        self.num_projs = nn.ModuleList([nn.Linear(1, d_emb) for _ in range(n_num)])
        self.cat_embs = nn.ModuleList([nn.Embedding(c, d_emb) for c in cat_cards])

    def forward(self, x_num, x_cat):
        num_t = [proj(x_num[:, i].unsqueeze(-1)) for i, proj in enumerate(self.num_projs)]
        cat_t = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embs)]
        return torch.stack(num_t + cat_t, dim=1)

class MultiTaskFTTransformer(nn.Module):
    def __init__(self, n_num, cat_cards, d_emb=32, n_layers=3, n_heads=8, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num, cat_cards, d_emb)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_emb))

        layer = nn.TransformerEncoderLayer(d_model=d_emb, nhead=n_heads, batch_first=True, dropout=dropout)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)

        # Cabeças de Saída (Task Heads)
        self.head_risk = nn.Linear(d_emb, 1)      # Classificação (Logits)
        self.head_credit = nn.Linear(d_emb, 1)    # Regressão (Scaled)
        self.head_annuity = nn.Linear(d_emb, 1)   # Regressão (Scaled)

    def forward(self, x_num, x_cat):
        z = self.tokenizer(x_num, x_cat)
        cls = self.cls_token.expand(z.shape[0], -1, -1)
        z = torch.cat([cls, z], dim=1)
        z = self.transformer(z)[:, 0, :] # Pega apenas a saída do CLS token
        return {'risk': self.head_risk(z), 'credit': self.head_credit(z), 'annuity': self.head_annuity(z)}

print("✅ Classe do Modelo FT-Transformer definida.")

In [ ]:
# Célula 7: Função de Perda Multi-Tarefa (Uncertainty Weights)
class MultiTaskLoss(nn.Module):
    def __init__(self, n_tasks=3):
        super().__init__()
        # Parâmetros de incerteza aprendíveis (log sigma^2)
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, preds, targets):
        l1 = F.binary_cross_entropy_with_logits(preds['risk'], targets['risk'])
        l2 = F.mse_loss(preds['credit'], targets['credit'])
        l3 = F.mse_loss(preds['annuity'], targets['annuity'])

        losses = [l1, l2, l3]
        total_loss = 0
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total_loss += precision * loss + self.log_vars[i]
        return total_loss

print("✅ Função de Perda Multi-Tarefa inicializada.")

In [ ]:
# Célula 8: Treinamento e Execução
# 1. Preparar Datasets e Dataloaders
class CreditDataset(Dataset):
    def __init__(self, xn, xc, ydf):
        self.xn, self.xc = torch.tensor(xn.values, dtype=torch.float32), torch.tensor(xc.values, dtype=torch.long)
        self.y = {k: torch.tensor(ydf[c].values, dtype=torch.float32).unsqueeze(-1)
                  for k, c in zip(['risk', 'credit', 'annuity'], ydf.columns)}
    def __len__(self): return len(self.xn)
    def __getitem__(self, i): return {'xn': self.xn[i], 'xc': self.xc[i], 'y': {k: v[i] for k, v in self.y.items()}}

train_loader = DataLoader(CreditDataset(X_train_num, X_train_cat, y_train), batch_size=1024, shuffle=True)

# 2. Inicializar Modelo
model = MultiTaskFTTransformer(len(num_cols), cat_cardinalities).to(device)
criterion = MultiTaskLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 3. Treino Inicial (10 épocas)
print("🚀 Iniciando Treinamento...")
for epoch in range(1, 11):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        xn, xc = batch['xn'].to(device), batch['xc'].to(device)
        targets = {k: v.to(device) for k, v in batch['y'].items()}

        optimizer.zero_grad()
        preds = model(xn, xc)
        loss = criterion(preds, targets)

        if not torch.isnan(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Proteção contra gradientes explosivos
            optimizer.step()
            epoch_loss += loss.item()

    print(f"Época {epoch}/10 | Loss: {epoch_loss/len(train_loader):.4f}")